# 5.2 · Regression & Inference That Doesn't Lie — companion lab

**Quant Notebook** · Unit 5 · Session 2 · [Read the session](https://georgevrs.github.io/quant-notebook/unit05-statistics-time-series/session02-regression-inference.html)

OLS alpha and beta, HAC standard errors, overlapping observations, the standard error of a
Sharpe ratio, bootstraps and multiple testing.

Run it top to bottom. Every number the session page quotes is recorded with `lab.record(...)`
and saved to `out/` by the last cell, so the page and this notebook can never disagree. All
returns here are already **excess** returns (synthetic, risk-free rate 0), so no further
risk-free adjustment is needed anywhere in this lab.

In [ ]:
# Colab or a fresh environment: install the course package (skipped when it is already installed).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("quantnb") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "quantnb @ git+https://github.com/georgevrs/quant-notebook@main"], check=True)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats as sstats
from scipy.signal import lfilter
from arch.bootstrap import MCS, SPA

import quantnb as qn
from quantnb import charts
from quantnb.returns import PERIODS_PER_YEAR as PPY
from quantnb.stats import annual_sharpe, batch_se, expected_max_sharpe, sharpe_se

lab = qn.Lab("5.2")  # seeds the random generators: every run gives the same numbers
rng_ols, rng_overlap, rng_sr, rng_noise, rng_boot = lab.rng.spawn(5)  # one stream per experiment

## 1 · OLS alpha and beta, classical vs HAC

A single-factor model with a KNOWN annual alpha and beta, built by hand so the truth is exact
(this is the pedagogical point here, so we do not reach for `quantnb.synth.factor_model_returns`,
which hides the construction behind random loadings). Daily shocks are i.i.d., so classical and
HAC standard errors should agree closely — the point of §2 below is what happens when they don't.

In [ ]:
TRUE_ALPHA, TRUE_BETA = 0.02, 1.20                 # annual alpha, beta on the factor
SIGMA_FACTOR, IDIO_VOL = 0.16, 0.25                # annual vols
YEARS_1 = 10
n1 = PPY * YEARS_1
factor = rng_ols.standard_normal(n1) * (SIGMA_FACTOR / np.sqrt(PPY))
idio = rng_ols.standard_normal(n1) * (IDIO_VOL / np.sqrt(PPY))
asset = TRUE_ALPHA / PPY + TRUE_BETA * factor + idio

X1 = sm.add_constant(factor)
ols1 = sm.OLS(asset, X1).fit()
hac1 = ols1.get_robustcov_results(cov_type="HAC", maxlags=5)

lab.record("true_alpha", TRUE_ALPHA)
lab.record("true_beta", TRUE_BETA)
lab.record("years1", YEARS_1)
lab.record("alpha_hat", float(ols1.params[0] * PPY))
lab.record("alpha_se_classical", float(ols1.bse[0] * PPY))
lab.record("alpha_se_hac", float(hac1.bse[0] * PPY))
lab.record("alpha_t_classical", float(ols1.tvalues[0]))
lab.record("alpha_p_classical", float(ols1.pvalues[0]))
lab.record("beta_hat", float(ols1.params[1]))
lab.record("beta_se_classical", float(ols1.bse[1]))
lab.record("beta_se_hac", float(hac1.bse[1]))
lab.record("beta_t_classical", float(ols1.tvalues[1]))
lab.record("beta_p_classical", float(ols1.pvalues[1]))
lab.record("r2_1", float(ols1.rsquared))
print(f"alpha_hat(annual)={ols1.params[0]*PPY:.2%} ± {ols1.bse[0]*PPY:.2%} (classical) / "
      f"{hac1.bse[0]*PPY:.2%} (HAC) · beta_hat={ols1.params[1]:.2f} ± {ols1.bse[1]:.2f}")

# Even ten years of daily data barely pins down alpha: check the truth sits inside a wide,
# honestly-computed band, not that the point estimate looks like 2%.
assert abs(ols1.params[0] * PPY - TRUE_ALPHA) < 4 * ols1.bse[0] * PPY
assert abs(ols1.params[1] - TRUE_BETA) < 4 * ols1.bse[1]
# i.i.d. shocks: classical and HAC standard errors should be close (within 15%)
assert abs(hac1.bse[1] / ols1.bse[1] - 1) < 0.15

## 2 · Overlapping observations: why the naive standard error lies

Build OVERLAPPING `K`-day returns from daily data (a rolling sum, stepped one day at a time) —
exactly what a monthly-rebalanced backtest or a "21-day forward return" regression does. Every
`K`-day window shares `K − 1` days with its neighbour, which manufactures autocorrelation in the
regression residuals even though the underlying daily shocks are i.i.d. We repeat the regression
on `N_SIMS` independent 5-year samples and compare the AVERAGE reported standard error (naive and
HAC) to the EMPIRICAL spread of the beta estimate across samples — the only honest measure of how
uncertain beta actually is.

In [ ]:
K = 21                      # overlap window ≈ one trading month
YEARS_2 = 5
n2 = PPY * YEARS_2
TRUE_BETA_2 = 0.80
N_SIMS_OVERLAP = 300

sims = rng_overlap.spawn(N_SIMS_OVERLAP)
beta_hat_o = np.empty(N_SIMS_OVERLAP)
se_classical_o = np.empty(N_SIMS_OVERLAP)
se_hac_o = np.empty(N_SIMS_OVERLAP)
for i, r in enumerate(sims):
    f = r.standard_normal(n2) * (SIGMA_FACTOR / np.sqrt(PPY))
    e = r.standard_normal(n2) * (IDIO_VOL / np.sqrt(PPY))
    daily = TRUE_BETA_2 * f + e
    # rolling K-day sums: every day advances the window by one day, so consecutive windows overlap
    f_roll = np.convolve(f, np.ones(K), mode="valid")
    s_roll = np.convolve(daily, np.ones(K), mode="valid")
    m = sm.OLS(s_roll, sm.add_constant(f_roll)).fit()
    m_hac = m.get_robustcov_results(cov_type="HAC", maxlags=K - 1)
    beta_hat_o[i] = m.params[1]
    se_classical_o[i] = m.bse[1]
    se_hac_o[i] = m_hac.bse[1]

empirical_sd_o = float(beta_hat_o.std(ddof=1))
mean_classical_o = float(se_classical_o.mean())
mean_hac_o = float(se_hac_o.mean())

lab.record("overlap_k", K)
lab.record("overlap_true_beta", TRUE_BETA_2)
lab.record("overlap_n_sims", N_SIMS_OVERLAP)
lab.record("overlap_years", YEARS_2)
lab.record("overlap_hac_maxlags", K - 1)
lab.record("overlap_empirical_sd", empirical_sd_o)
lab.record("overlap_mean_classical_se", mean_classical_o)
lab.record("overlap_mean_hac_se", mean_hac_o)
lab.record("overlap_ratio_classical", mean_classical_o / empirical_sd_o)
lab.record("overlap_ratio_hac", mean_hac_o / empirical_sd_o)
print(f"empirical SD of beta_hat = {empirical_sd_o:.3f} · mean naive SE = {mean_classical_o:.3f} "
      f"({mean_classical_o/empirical_sd_o:.0%} of the truth) · mean HAC SE = {mean_hac_o:.3f} "
      f"({mean_hac_o/empirical_sd_o:.0%} of the truth)")

# the naive SE must substantially understate the true sampling spread of beta...
assert mean_classical_o / empirical_sd_o < 0.5
# ...while HAC with K-1 lags gets most (but need not get all) of the way there
assert 0.7 < mean_hac_o / empirical_sd_o < 1.05

## 3 · The standard error of the Sharpe ratio, with an autocorrelation correction

Lo (2002)'s i.i.d. result is `SE(SR_p) = sqrt((1 + SR_p^2/2) / n)` for the per-period Sharpe
ratio over `n` periods (`quantnb.stats.sharpe_se`; met already in Sessions 1.4 and 2.3). When
returns are autocorrelated the true sampling variance is inflated by the SAME long-run-variance
factor that HAC uses for a mean:

  Psi = 1 + 2 * sum_{k=1}^{n-1} (1 - k/n) * rho_k

so `SE_corrected(SR_p) ~ sqrt(Psi) * SE_iid(SR_p)`. We check this on synthetic AR(1) daily
returns with a KNOWN autocorrelation `RHO` (mild — the kind produced by monthly-smoothed or
stale pricing; see Getmansky, Lo & Makarov, 2004), so `Psi` can be computed exactly rather than
estimated.

In [ ]:
def ar1_returns(n, mu_annual, sigma_annual, rho, rng, periods_per_year=PPY):
    """Stationary AR(1) daily excess returns with the given ANNUAL mean and volatility."""
    mu = mu_annual / periods_per_year
    sigma = sigma_annual / np.sqrt(periods_per_year)
    eps = rng.standard_normal(n)
    centered = lfilter([sigma * np.sqrt(1 - rho ** 2)], [1, -rho], eps)
    return mu + centered


def psi_factor(rho, q):
    """Long-run-variance inflation factor for an AR(1) process over q periods (Lo, 2002)."""
    k = np.arange(1, q)
    return float(1.0 + 2.0 * np.sum((1 - k / q) * rho ** k))


TRUE_SR, RHO, YEARS_3 = 0.80, 0.15, 3
TRUE_SIGMA_3 = 0.20
n3 = PPY * YEARS_3
N_SIMS_SR = 2_000
psi = psi_factor(RHO, n3)

sims_sr = rng_sr.spawn(N_SIMS_SR)
sr_hat = np.empty(N_SIMS_SR)
naive_se_sr = np.empty(N_SIMS_SR)
for i, r in enumerate(sims_sr):
    x = ar1_returns(n3, TRUE_SR * TRUE_SIGMA_3, TRUE_SIGMA_3, RHO, r)
    sr_p = x.mean() / x.std(ddof=1)
    sr_hat[i] = sr_p * np.sqrt(PPY)
    naive_se_sr[i] = sharpe_se(sr_hat[i], n3 / PPY)

empirical_sd_sr = float(sr_hat.std(ddof=1))
mean_naive_sr = float(naive_se_sr.mean())
mean_corrected_sr = mean_naive_sr * np.sqrt(psi)

lab.record("sr_true", TRUE_SR)
lab.record("sr_rho", RHO)
lab.record("sr_years", YEARS_3)
lab.record("sr_n_sims", N_SIMS_SR)
lab.record("sr_psi", psi)
lab.record("sr_psi_asymptotic", (1 + RHO) / (1 - RHO))
lab.record("sr_mean_hat", float(sr_hat.mean()))
lab.record("sr_empirical_sd", empirical_sd_sr)
lab.record("sr_mean_naive_se", mean_naive_sr)
lab.record("sr_mean_corrected_se", mean_corrected_sr)
lab.record("sr_ratio_naive", mean_naive_sr / empirical_sd_sr)
lab.record("sr_ratio_corrected", mean_corrected_sr / empirical_sd_sr)
print(f"Psi={psi:.3f} · empirical SD={empirical_sd_sr:.3f} · naive SE={mean_naive_sr:.3f} "
      f"({mean_naive_sr/empirical_sd_sr:.0%}) · corrected SE={mean_corrected_sr:.3f} "
      f"({mean_corrected_sr/empirical_sd_sr:.0%})")

assert abs(psi - (1 + RHO) / (1 - RHO)) < 0.01          # finite-sample Psi ≈ its asymptotic limit here
assert abs(sr_hat.mean() - TRUE_SR) < 4 * batch_se(lambda b: b.mean(), sr_hat.reshape(-1, 1), n_batches=40)
# theory: E[naive/empirical] -> 1/sqrt(Psi) (naive ignores the correction), E[corrected/empirical] -> 1
assert 0.75 < mean_naive_sr / empirical_sd_sr < 1 / np.sqrt(psi) + 0.08     # naive SE understates
assert 0.90 < mean_corrected_sr / empirical_sd_sr < 1.15                    # the correction gets close

## 4 · Multiple testing: FWER, FDR and the False Strategy Theorem

`N_TRIALS` strategies, all pure noise (zero true skill), each tested at `ALPHA = 5%`. Some will
clear the bar by chance alone — that is what a p-value promises, not a bug.

In [ ]:
N_TRIALS, YEARS_MT, ALPHA_MT = 200, 5, 0.05
n_mt = PPY * YEARS_MT
noise = rng_noise.standard_normal((n_mt, N_TRIALS)) * 0.01
tstats = noise.mean(axis=0) / noise.std(axis=0, ddof=1) * np.sqrt(n_mt)
pvals = 2 * (1 - sstats.norm.cdf(np.abs(tstats)))

naive_discoveries = int((pvals < ALPHA_MT).sum())
fwer = 1 - (1 - ALPHA_MT) ** N_TRIALS
bonferroni_thresh = ALPHA_MT / N_TRIALS
bonferroni_discoveries = int((pvals < bonferroni_thresh).sum())

order = np.argsort(pvals)
ranked = pvals[order]
bh_thresh = (np.arange(1, N_TRIALS + 1) / N_TRIALS) * ALPHA_MT
below = ranked <= bh_thresh
bh_discoveries = int(np.max(np.where(below)) + 1) if below.any() else 0

best_i = int(np.argmax(tstats))
best_p = float(pvals[best_i])
best_sharpe = float(annual_sharpe(noise[:, best_i], periods_per_year=PPY))
max_sharpe_all = float(annual_sharpe(noise, axis=0, periods_per_year=PPY).max())
theory_max_sharpe = expected_max_sharpe(N_TRIALS, YEARS_MT)

lab.record("mt_n_trials", N_TRIALS)
lab.record("mt_years", YEARS_MT)
lab.record("mt_alpha", ALPHA_MT)
lab.record("mt_naive_discoveries", naive_discoveries)
lab.record("mt_expected_discoveries", ALPHA_MT * N_TRIALS)
lab.record("mt_fwer", fwer)
lab.record("mt_bonferroni_thresh", bonferroni_thresh)
lab.record("mt_bonferroni_discoveries", bonferroni_discoveries)
lab.record("mt_bh_discoveries", bh_discoveries)
lab.record("mt_best_p", best_p)
lab.record("mt_best_sharpe", best_sharpe)
lab.record("mt_max_sharpe_all", max_sharpe_all)
lab.record("mt_theory_max_sharpe", theory_max_sharpe)
print(f"naive discoveries {naive_discoveries}/{N_TRIALS} · FWER={fwer:.1%} · "
      f"Bonferroni discoveries={bonferroni_discoveries} · BH discoveries={bh_discoveries}")
print(f"best trial: p={best_p:.4f}, annualised Sharpe={best_sharpe:.2f} "
      f"(theory max over {N_TRIALS} trials ≈ {theory_max_sharpe:.2f})")

binom_se = np.sqrt(N_TRIALS * ALPHA_MT * (1 - ALPHA_MT))
assert abs(naive_discoveries - ALPHA_MT * N_TRIALS) < 4 * binom_se   # ≈ alpha fraction, by construction
assert fwer > 0.99                                                  # with 200 independent looks, near-certain
assert bonferroni_discoveries <= 2 and bh_discoveries <= 2           # both correctly stay near zero on pure noise
assert best_p < 0.01                                                 # the best of 200 looks individually "significant"
assert abs(max_sharpe_all - theory_max_sharpe) / theory_max_sharpe < 0.15  # matches the 1.4 False Strategy Theorem

## 5 · White's SPA test and Hansen's model confidence set

The same 200 noise trials, now judged properly: does ANY of them really beat doing nothing (a
zero benchmark), once you account for having looked 200 times? `arch.bootstrap` implements both
procedures on a stationary bootstrap (Politis & Romano, 1994) — we do not reimplement them.

In [ ]:
losses_benchmark = np.zeros(n_mt)      # "loss" of a zero-return benchmark
losses_models = -noise                 # loss = -return, so a higher return is a lower loss

spa = SPA(losses_benchmark, losses_models, reps=500, seed=rng_boot.spawn(1)[0])
spa.compute()
spa_p_consistent = float(spa.pvalues["consistent"])

mcs = MCS(losses_models, size=0.10, reps=500, seed=rng_boot.spawn(1)[0])
mcs.compute()
mcs_included = len(mcs.included)
mcs_excluded = len(mcs.excluded)

lab.record("spa_reps", 500)
lab.record("spa_p_consistent", spa_p_consistent)
lab.record("mcs_size", 0.10)
lab.record("mcs_reps", 500)
lab.record("mcs_included", mcs_included)
lab.record("mcs_excluded", mcs_excluded)
print(f"SPA consistent p-value = {spa_p_consistent:.3f} (best trial's own p-value was {best_p:.4f}) · "
      f"MCS keeps {mcs_included}/{N_TRIALS} models, excludes {mcs_excluded}")

assert spa_p_consistent > 0.10           # correctly fails to reject "no model beats the benchmark"
assert mcs_included / N_TRIALS > 0.8     # correctly keeps almost all models — none is truly distinguishable

## 6 · Charts for the session page

In [ ]:
lab.chart("ols_scatter", charts.scatter_chart(
    factor, asset, title="Daily asset vs factor excess returns, with the OLS fit",
    x_fmt=charts.fmt_pct(1), y_fmt=charts.fmt_pct(1), fit_line=True, max_points=600,
    x_label="factor daily return", y_label="asset daily return"))

lab.chart("overlap_se", charts.column_chart(
    ["Naive OLS", "HAC (Newey-West)", "Empirical (Monte Carlo)"],
    [mean_classical_o, mean_hac_o, empirical_sd_o],
    title=f"SE of β on {K}-day overlapping returns — naive vs HAC vs the truth",
    y_fmt=charts.fmt_num(2), roles=["loss", "strategy", "benchmark"]))

lab.chart("sharpe_se", charts.column_chart(
    ["Naive (i.i.d.)", "Autocorrelation-corrected", "Empirical (Monte Carlo)"],
    [mean_naive_sr, mean_corrected_sr, empirical_sd_sr],
    title="SE of the annualised Sharpe ratio under mild autocorrelation",
    y_fmt=charts.fmt_num(2), roles=["loss", "strategy", "benchmark"]))

lab.chart("noise_tstats", charts.histogram(
    tstats, title=f"t-statistics of {N_TRIALS} pure-noise ‘strategies’",
    x_fmt=charts.fmt_num(1), bins=40))

n_grid = np.array([1, 5, 10, 20, 50, 100, 150, 200])
fwer_grid = 1 - (1 - ALPHA_MT) ** n_grid
lab.chart("fwer_curve", charts.line_chart(
    [charts.Series("FWER at α = 5%", pd.Series(fwer_grid, index=n_grid), role="loss", area=True)],
    title="Chance of at least one false positive, by number of strategies tried",
    y_fmt=charts.fmt_pct(0), y_min=0.0, y_max=1.0))

In [ ]:
lab.save()